# 01 · Setup datasets — Kaggle → Hugging Face

Downloads the PaySim dataset from Kaggle and generates the custom dataset, then
publishes both as a **private HF dataset repo** the other notebooks read from.

**Important:** The Kaggle SDK runs `authenticate()` at import time, so we must
write the credentials file **before** the first `import kaggle`. The cells
below are ordered accordingly — run them in sequence.

## 1. Write Kaggle + HF credentials from Space secrets

Required secrets (Space → Settings → Variables and secrets):
- `HF_TOKEN` — HF write token
- one of:
  - `KAGGLE_ACCESS_TOKEN` (new OAuth-style token starting with `KGAT_`), or
  - `KAGGLE_USERNAME` + `KAGGLE_KEY` (legacy)

This cell must run **before** any `import kaggle`.

In [ ]:
import os, json
from pathlib import Path

HF_TOKEN = os.environ['HF_TOKEN']
HF_USERNAME = os.environ.get('HF_USERNAME', '').strip()

kdir = Path.home() / '.kaggle'
kdir.mkdir(exist_ok=True)

kaggle_access_token = os.environ.get('KAGGLE_ACCESS_TOKEN', '').strip()
kaggle_username     = os.environ.get('KAGGLE_USERNAME', '').strip()
kaggle_key          = os.environ.get('KAGGLE_KEY', '').strip()

if kaggle_access_token:
    (kdir / 'access_token').write_text(kaggle_access_token + '\n')
    os.chmod(kdir / 'access_token', 0o600)
    print('Wrote', kdir / 'access_token', '(KAGGLE_ACCESS_TOKEN, OAuth format)')
elif kaggle_username and kaggle_key:
    (kdir / 'kaggle.json').write_text(json.dumps({'username': kaggle_username, 'key': kaggle_key}))
    os.chmod(kdir / 'kaggle.json', 0o600)
    print('Wrote', kdir / 'kaggle.json', '(legacy username + key)')
else:
    raise RuntimeError(
        'No Kaggle credentials in env. Set KAGGLE_ACCESS_TOKEN, or KAGGLE_USERNAME + KAGGLE_KEY, '
        'on this Space (Settings → Variables and secrets).'
    )

print('HF user:', HF_USERNAME or '(will resolve from token)')

## 2. Check libraries — uses `importlib.util.find_spec` so we never trigger Kaggle's import-time auth

In [ ]:
import sys, subprocess, importlib.util
for pkg in ('huggingface_hub', 'kaggle', 'pandas', 'tqdm'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('deps OK')

## 3. Resolve HF username if not set, and define target dataset repo

In [ ]:
from huggingface_hub import whoami
if not HF_USERNAME:
    HF_USERNAME = whoami(token=HF_TOKEN)['name']
DATASET_REPO = f'{HF_USERNAME}/fraud-detection-datasets'
print('Dataset target:', DATASET_REPO)

## 4. Download PaySim from Kaggle

Now safe to import `kaggle` because the credentials file exists.

In [ ]:
import shutil, zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path('/tmp/fraudguard_data'); WORK.mkdir(parents=True, exist_ok=True)

def download_slug(slug, dest):
    api = KaggleApi(); api.authenticate()
    tmp = WORK / slug.replace('/', '_'); tmp.mkdir(parents=True, exist_ok=True)
    print(f'[kaggle] downloading {slug}…')
    api.dataset_download_files(slug, path=str(tmp), quiet=False, unzip=False)
    for zp in tmp.glob('*.zip'):
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(tmp)
        zp.unlink()
    csvs = sorted(tmp.glob('**/*.csv'), key=lambda p: p.stat().st_size, reverse=True)
    if not csvs:
        raise RuntimeError(f'No CSVs found in {slug}')
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(csvs[0]), dest)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f'  saved -> {dest} ({dest.stat().st_size/1e6:.1f} MB)')
    return dest

paysim_path = download_slug('ealaxi/paysim1', WORK / 'paysim.csv')

## 5. Generate the custom mixed-feature dataset (51k rows, ~5% fraud)

In [ ]:
import numpy as np
import pandas as pd

def make_custom(n=51_000, fraud_rate=0.05, seed=7):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        'Transaction_ID': [f'T{i+1}' for i in range(n)],
        'User_ID': rng.integers(1, 10_000, n),
        'Transaction_Amount': rng.exponential(scale=2500, size=n),
        'Transaction_Type': rng.choice(['ATM Withdrawal','Bill Payment','Online Purchase','POS Payment','Bank Transfer'], n),
        'Time_of_Transaction': rng.uniform(0, 23, n).round(0),
        'Device_Used': rng.choice(['Desktop','Mobile','Tablet','Unknown Device'], n, p=[0.31,0.31,0.30,0.08]),
        'Location': rng.choice(['Boston','Chicago','Houston','Los Angeles','Miami','New York','San Francisco','Seattle'], n),
        'Previous_Fraudulent_Transactions': rng.integers(0, 6, n),
        'Account_Age': rng.integers(1, 1500, n),
        'Number_of_Transactions_Last_24H': rng.integers(0, 25, n),
        'Payment_Method': rng.choice(['Credit Card','Debit Card','Net Banking','UPI','Invalid Method'], n, p=[0.23,0.24,0.23,0.23,0.07]),
    })
    mask = rng.random(n) < fraud_rate
    df.loc[mask, 'Previous_Fraudulent_Transactions'] = rng.integers(3, 8, mask.sum())
    df.loc[mask, 'Account_Age'] = rng.integers(1, 30, mask.sum())
    df.loc[mask, 'Transaction_Amount'] = rng.uniform(5_000, 20_000, mask.sum())
    df.loc[mask, 'Payment_Method'] = rng.choice(['Invalid Method','Credit Card'], mask.sum(), p=[0.5,0.5])
    df['Fraudulent'] = mask.astype(int)
    return df

custom_df = make_custom()
custom_path = WORK / 'custom.csv'
custom_df.to_csv(custom_path, index=False)
print(f'Custom dataset: {custom_df.shape}, fraud rate {custom_df.Fraudulent.mean():.3%}')

## 6. Push both CSVs to a private HF dataset repo

In [ ]:
from huggingface_hub import HfApi, create_repo

create_repo(repo_id=DATASET_REPO, repo_type='dataset', token=HF_TOKEN, private=True, exist_ok=True)
api = HfApi()
for src, dst in [(paysim_path, 'paysim.csv'), (custom_path, 'custom.csv')]:
    print(f'Uploading {src.name} -> {DATASET_REPO}:{dst}')
    api.upload_file(
        path_or_fileobj=str(src), path_in_repo=dst,
        repo_id=DATASET_REPO, repo_type='dataset', token=HF_TOKEN,
        commit_message=f'Upload {dst}',
    )
print('All uploaded ✔')
print(f'View: https://huggingface.co/datasets/{DATASET_REPO}')